In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3

mkdir: cannot create directory ‘/home/ec2-user/tmpdir’: File exists
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.


In [2]:
# Prepare dataset
%cd ~/SageMaker/
! rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/puneet6060/intel-image-classification"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/
!mv ./dataset/seg_train/seg_train ./dataset/train
!mv ./dataset/seg_test/seg_test ./dataset/test
!rm -r ./dataset/seg_* dataset.zip

%cd ./dataset/
!zip -rq ../dataset.zip .
%cd ..

/home/ec2-user/SageMaker
/home/ec2-user/SageMaker/dataset
/home/ec2-user/SageMaker


In [3]:
# Import pip packages
import pandas as pd
import numpy as np
import torch
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from IPython.display import display
from sagemaker.inputs import TrainingInput
from sagemaker.s3 import S3Uploader

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [5]:
# Upload dataset zip file to S3
inputs = S3Uploader.upload("dataset.zip", "s3://{}/{}".format(bucket, prefix))

In [6]:
# Find mean, std
_dataset = datasets.ImageFolder(root='./dataset/train', transform=transforms.ToTensor())
_loader  = DataLoader(_dataset)

mean   = torch.zeros(3)
sq_mean = torch.zeros(3)
n_pixels = 0

with torch.no_grad():
    for images, _ in _loader:
        b, c, h, w = images.shape
        n_pixels += b * h * w
        mean     += images.sum(dim=[0, 2, 3])
        sq_mean  += (images ** 2).sum(dim=[0, 2, 3])

mean  /= n_pixels
std    = (sq_mean / n_pixels - mean ** 2).sqrt()

mean, std

(tensor([0.4302, 0.4575, 0.4538]), tensor([0.2694, 0.2679, 0.2983]))

In [11]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",       # DDP 버전 스크립트 (단일 GPU는 train.py)
    framework_version="2.8.0",
    py_version="py312",
    instance_type="ml.g5.12xlarge",   # GPU 4개 (단일 GPU는 ml.g5.xlarge)
    instance_count=1,                 # 2 이상으로 올리면 멀티 인스턴스 분산도 그대로 동작
    distribution={"torch_distributed": {"enabled": True}},  # torchrun으로 GPU당 1프로세스 실행
    environment={"FI_EFA_FORK_SAFE": "1"},  # DataLoader 워커(fork)와 EFA 충돌 방지
    output_path="s3://{}/{}/output/".format(bucket, prefix),
    checkpoint_s3_uri="s3://{}/{}/checkpoint/".format(bucket, prefix),
    role=role,
    hyperparameters = {
        'epochs': 20,
        'batch_size': 64,  # GPU당 배치 크기 (유효 배치 = 64 * GPU 수)
        'lr': 0.001,
        'workers': 8,  # GPU(프로세스)당 DataLoader 워커 수 (총 8 * 4 = 32, vCPU 48개 내)
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"Train Loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"Train Acc: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"Test Loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"Test Acc: ([0-9\.]+)"},
    ],
)

estimator.fit({
    "train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix)),  # /opt/ml/input/data/train
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2026-07-03-12-30-59-944


2026-07-03 12:31:00 Starting - Starting the training job...
2026-07-03 12:31:19 Pending - Training job waiting for capacity...
2026-07-03 12:31:44 Pending - Preparing the instances for training...
2026-07-03 12:32:35 Downloading - Downloading the training image........bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 575.57.08
Current installed NVIDIA driver version is 570.211.01
Adding CUDA compat to LD_LIBRARY_PATH
/usr/local/cuda/compat:/usr/local/lib:/opt/amazon/ofi-nccl/lib/x86_64-linux-gnu:/opt/amazon/openmpi/lib:/opt/amazon/efa/lib:/usr/local/cuda/lib64:/usr/local/lib:/usr/local/cuda/lib64:/opt/amazon/ofi-nccl/lib:/opt/amazon/efa/lib:/opt/amazon/openmpi/lib:/usr/local/cuda/lib64
2026-07-03 12:33:43,827 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-07-03 12:33:43,882 sagemaker-training-toolkit INFO 